# BioSkills Lab — Chapter 9
## Neural Networks and Autoencoders on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **GPU recommended** — Runtime > Change runtime type > T4 GPU
> Self-contained: loads TCGA data automatically if Chapter 7 is not already in memory.

In [ ]:
!pip install -q torch scikit-learn numpy pandas matplotlib requests

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
print(f'GPU: {torch.cuda.is_available()}')

In [ ]:
try:
    _ = X_train_pca
    print('Chapter 7 data already loaded')
except NameError:
    BASE = 'https://www.cbioportal.org/api'
    STUDIES = {'BRCA':'brca_tcga','LUAD':'luad_tcga','KIRC':'kirc_tcga','PRAD':'prad_tcga','COAD':'coad_tcga'}
    GENE_IDS = [672,675,7157,4609,5728,1029,595,896,2064,2065,1956,3320,4292,5594,5595,5604,5605,207,208,
                10000,4893,4914,5290,6662,6663,3845,4763,4764,7422,7424,7525,7528,1499,4005,7409,8503,
                9020,2099,2100,2263,2534,3169,3236,4254,4255,5156,5163,5241,5579,6009,6196,7163,7490,
                7849,8379,9054,9500,10257,10319,10488,11200,23533,51176,55294,1026,1027,1028,1030,1031,
                1032,1869,1870,4616,4619,8317,8318,701,994,999,1000,4193,4831,5925,6237,7153,7538]
    all_frames = []
    for cancer, study in STUDIES.items():
        print(f'  Fetching {cancer}...', end=' ')
        try:
            r = requests.get(f'{BASE}/studies/{study}/samples', params={'pageSize':150}, timeout=30)
            samples = [s['sampleId'] for s in r.json()[:150]]
            r2 = requests.post(f'{BASE}/molecular-profiles/{study}_rna_seq_v2_mrna/molecular-data/fetch',
                               params={'projection':'SUMMARY'},
                               json={'sampleIds':samples,'entrezGeneIds':GENE_IDS}, timeout=120)
            if r2.status_code != 200: print('failed'); continue
            df = pd.DataFrame([{'sample':d['sampleId'],'gene':d['entrezGeneId'],'value':d['value']}
                               for d in r2.json() if d.get('value') is not None])
            pivot = df.pivot_table(index='sample',columns='gene',values='value',aggfunc='first')
            pivot['cancer_type'] = cancer
            all_frames.append(pivot)
            print(f'{len(pivot)} samples')
        except Exception as e: print(f'error: {e}')
    if not all_frames:
        np.random.seed(42); n,g=150,500
        ct_list=list(STUDIES.keys()); blocks,labs=[],[]
        for i,ct in enumerate(ct_list):
            X=np.random.randn(n,g)*0.5; X[:,i*100:(i+1)*100]+=3
            blocks.append(X); labs.extend([ct]*n)
        expr=pd.DataFrame(np.vstack(blocks),columns=[f'g{i}' for i in range(g)]); expr['cancer_type']=labs
    else: expr=pd.concat(all_frames)
    y_raw=expr['cancer_type'].values
    X_raw=np.nan_to_num(expr.drop(columns=['cancer_type']).values.astype(np.float32))
    le=LabelEncoder(); y_enc=le.fit_transform(y_raw)
    X_train,X_test,y_train,y_test=train_test_split(X_raw,y_enc,test_size=0.2,random_state=42,stratify=y_enc)
    scaler=StandardScaler()
    X_train_s=scaler.fit_transform(X_train); X_test_s=scaler.transform(X_test)
    pca=PCA(n_components=min(50,X_train_s.shape[1]-1))
    X_train_pca=pca.fit_transform(X_train_s); X_test_pca=pca.transform(X_test_s)
    print(f'Data ready. Shape: {X_train_pca.shape}')

## Neural Network Classifier

In [ ]:
class CancerClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim,hidden_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2,output_dim)
        )
    def forward(self, x): return self.network(x)

In [ ]:
n_classes = len(np.unique(y_train))
model = CancerClassifier(X_train_pca.shape[1], 256, n_classes)
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_pca), torch.LongTensor(y_train)), batch_size=128, shuffle=True)

train_accs, test_accs = [], []
for epoch in range(50):
    model.train()
    for xb, yb in loader:
        optimiser.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimiser.step()
    model.eval()
    with torch.no_grad():
        tr = (model(torch.FloatTensor(X_train_pca)).argmax(1)==torch.LongTensor(y_train)).float().mean().item()
        te = (model(torch.FloatTensor(X_test_pca)).argmax(1)==torch.LongTensor(y_test)).float().mean().item()
    train_accs.append(tr); test_accs.append(te)
    if (epoch+1)%10==0: print(f'Epoch {epoch+1}: train={tr:.3f}, test={te:.3f}')

In [ ]:
plt.plot(train_accs, label='Train'); plt.plot(test_accs, label='Test')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Neural Network Learning Curves'); plt.legend(); plt.show()

## Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim,256),nn.ReLU(),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim,128),nn.ReLU(),nn.Linear(128,256),nn.ReLU(),nn.Linear(256,input_dim))
    def forward(self, x): z=self.encoder(x); return self.decoder(z), z
    def encode(self, x): return self.encoder(x)

ae = Autoencoder(X_train_pca.shape[1], 32)
ae_opt = optim.Adam(ae.parameters(), lr=1e-3)
ae_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_pca)), batch_size=128, shuffle=True)
for epoch in range(100):
    ae.train()
    for (xb,) in ae_loader:
        ae_opt.zero_grad(); xr,_=ae(xb); loss=nn.MSELoss()(xr,xb); loss.backward(); ae_opt.step()
    if (epoch+1)%25==0: print(f'Epoch {epoch+1}: recon_loss={loss.item():.4f}')

In [ ]:
ae.eval()
with torch.no_grad():
    Z_train=ae.encode(torch.FloatTensor(X_train_pca)).numpy()
    Z_test =ae.encode(torch.FloatTensor(X_test_pca)).numpy()
lr_ae=LogisticRegression(max_iter=1000)
lr_ae.fit(Z_train,y_train)
print(f'Autoencoder + LR accuracy: {accuracy_score(y_test, lr_ae.predict(Z_test)):.3f}')